# Ground-State-Only `a`, `b`, and $K_0$ Analysis

This notebook uses the clean `ground_state_ab_k0` modules to compute only the ground-state quantities.

Included:
- parameter fits for `a` and `b`
- incompressibility `K_0`
- simple tables and summary plots

Excluded:
- finite-temperature isotherms
- critical-point quantities such as `T_c` and `n_c`


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
project_dir = None

for candidate in [cwd, cwd.parent, cwd / "ground_state_ab_k0"]:
    if (candidate / "src" / "ground_state_ab_k0").exists():
        project_dir = candidate
        break

if project_dir is None:
    raise RuntimeError("Could not locate ground_state_ab_k0/src.")

src_dir = project_dir / "src"
results_dir = project_dir / "results"
results_dir.mkdir(exist_ok=True)

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from ground_state_ab_k0 import DEFAULT_ALPHA_LIST, DEFAULT_C_LIST, compute_ground_state_point, compute_model_family
from ground_state_ab_k0.reporting import results_to_records


In [ ]:
alpha_list = DEFAULT_ALPHA_LIST
c_list = DEFAULT_C_LIST

base_results = [
    compute_ground_state_point("vdw"),
    compute_ground_state_point("rks"),
    compute_ground_state_point("pr"),
]
base_results = [result for result in base_results if result is not None]

clausius_results = compute_model_family("clausius", c_list)
dieterici_results = compute_model_family("dieterici", alpha_list)


In [ ]:
base_df = pd.DataFrame(results_to_records(base_results))
clausius_df = pd.DataFrame(results_to_records(clausius_results, parameter_name="c"))
dieterici_df = pd.DataFrame(results_to_records(dieterici_results, parameter_name="alpha"))

combined_df = pd.concat([base_df, clausius_df, dieterici_df], ignore_index=True)

base_df.to_csv(results_dir / "base_models_ground_state.csv", index=False)
clausius_df.to_csv(results_dir / "clausius_ground_state.csv", index=False)
dieterici_df.to_csv(results_dir / "dieterici_ground_state.csv", index=False)
combined_df.to_csv(results_dir / "ground_state_ab_k0_results.csv", index=False)

display(base_df[["model", "a", "b", "K0", "binding", "n_sat", "p_sat"]])
display(clausius_df[["parameter_value", "a", "b", "K0", "binding"]])
display(dieterici_df[["parameter_value", "a", "b", "K0", "binding"]])


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(base_df["model"], base_df["K0"], color=["black", "red", "blue"], s=80)
ax.set_ylabel(r"$K_0$ [MeV]")
ax.set_title("Base-Model Incompressibility")
fig.tight_layout()
fig.savefig(results_dir / "base_models_k0.png", dpi=300)
plt.show()


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(10, 10), sharex="col")

axes[0, 0].plot(clausius_df["parameter_value"], clausius_df["a"], marker="o", color="magenta")
axes[1, 0].plot(clausius_df["parameter_value"], clausius_df["b"], marker="o", color="magenta")
axes[2, 0].plot(clausius_df["parameter_value"], clausius_df["K0"], marker="o", color="magenta")

axes[0, 1].plot(dieterici_df["parameter_value"], dieterici_df["a"], marker="o", color="green")
axes[1, 1].plot(dieterici_df["parameter_value"], dieterici_df["b"], marker="o", color="green")
axes[2, 1].plot(dieterici_df["parameter_value"], dieterici_df["K0"], marker="o", color="green")

axes[0, 0].set_title("Clausius sweep")
axes[0, 1].set_title("Dieterici sweep")

axes[0, 0].set_ylabel("a")
axes[1, 0].set_ylabel("b")
axes[2, 0].set_ylabel(r"$K_0$ [MeV]")

axes[2, 0].set_xlabel("c")
axes[2, 1].set_xlabel(r"$\alpha$")

for axis in axes.flat:
    axis.grid(alpha=0.25)

fig.tight_layout()
fig.savefig(results_dir / "ground_state_family_summary.png", dpi=300)
plt.show()
